# Heart Disease Classification - PyTorch Experiment Framework

**Dataset:** Heart Disease UCI (`ineubytes/heart-disease-dataset`) - 1,025 raw rows, 302 unique patients, 13 features  
**Task:** Binary classification - predict presence of heart disease  
**Design:** Two-phase, multi-seed stratified cross-validation
- **Phase 1:** All regularizers × fixed optimizer - select best regularizer
- **Phase 2:** All optimizers × Phase 1 winner - select best optimizer
- **Final:** Train the selected configuration on all development data with SWA; evaluate once on untouched test data

Every candidate uses 5 folds × 3 seeds. Configuration lists and resampling parameters are defined once below.

## 0. Setup & Imports

In [1]:
import os, warnings, random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using device: cuda
GPU: NVIDIA RTX PRO 1000 Blackwell Generation Laptop GPU


## 1. Configuration

> **Single source of truth.** Modify `REGULARIZERS` or `OPTIMIZERS` here - every downstream cell adapts automatically.

In [ ]:
# ============================================================
#  REGULARIZER LIST
#  Each dict must have 'name' (display label) and 'type'
#  Supported types: none | l1 | l2 | dropout | label_smooth | bn_dropout
# ============================================================
REGULARIZERS = [
    {'name': 'Baseline',          'type': 'none'},
    {'name': 'L1',                'type': 'l1',           'l1_lambda': 1e-4},
    {'name': 'L2 (WeightDecay)',  'type': 'l2',           'weight_decay': 1e-4},
    {'name': 'Dropout',           'type': 'dropout',      'rate': 0.4},
    {'name': 'LabelSmoothing',    'type': 'label_smooth', 'smoothing': 0.1},
    {'name': 'BatchNorm+Dropout', 'type': 'bn_dropout',   'rate': 0.3},
]

# ============================================================
#  OPTIMIZER LIST
#  Each dict must have 'name' (display label) and 'type'
#  Supported types: sgd | adam | adamw | rmsprop
# ============================================================
# AdamW uses torch's default weight_decay=1e-2. At 1e-4 its decoupled decay is
# ~1e-7 per step and the optimizer becomes numerically indistinguishable from Adam.
OPTIMIZERS = [
    {'name': 'SGD',          'type': 'sgd',     'lr': 0.01},
    {'name': 'SGD+Momentum', 'type': 'sgd',     'lr': 0.01, 'momentum': 0.9},
    {'name': 'Adam',         'type': 'adam',    'lr': 1e-3},
    {'name': 'AdamW',        'type': 'adamw',   'lr': 1e-3, 'weight_decay': 1e-2},
    {'name': 'RMSprop',      'type': 'rmsprop', 'lr': 1e-3},
]

PHASE1_OPTIMIZER_NAME = 'Adam'

# ============================================================
#  RESAMPLING & TRAINING
# ============================================================
CV_FOLDS       = 5      # Cross Validation folds (StratifiedKFold)
CV_SEEDS       = [42, 123, 2026]
TEST_SIZE      = 0.15
# 300 gives plain SGD room to converge; without it SGD hits the cap while the
# Adam-family optimizers early-stop far sooner, which would confound the comparison.
EPOCHS         = 300
BATCH_SIZE     = 32
EARLY_STOP_PAT = 20     # stop if val_loss does not improve for N epochs
SWA_START_FRAC = 0.75   # SWA averaging begins at this fraction of EPOCHS

HIDDEN1 = 64
HIDDEN2 = 32

# CV means drive model selection; corresponding cv_std_* columns quantify stability.
METRIC_MAP = {'cv_roc_auc': 'ROC-AUC', 'cv_f1': 'F1-Score', 'cv_accuracy': 'Accuracy'}

# ============================================================
#  DATA LOADING MODE
# ============================================================
USE_LOCAL_CSV = False

print(f'Regularizers: {[r["name"] for r in REGULARIZERS]}')
print(f'Optimizers:   {[o["name"] for o in OPTIMIZERS]}')
print(f'Phase 1 optimizer: {PHASE1_OPTIMIZER_NAME}')
print(f'CV design: {CV_FOLDS} folds × {len(CV_SEEDS)} seeds = {CV_FOLDS * len(CV_SEEDS)} fits/config')
print(f'Untouched test fraction: {TEST_SIZE:.0%}')
print(f'Total model-selection fits: {(len(REGULARIZERS) + len(OPTIMIZERS)) * CV_FOLDS * len(CV_SEEDS)}')
print(f'Epoch budget: {EPOCHS} (early stopping patience {EARLY_STOP_PAT})')
print(f'USE_LOCAL_CSV: {USE_LOCAL_CSV}')


Regularizers: ['Baseline', 'L1', 'L2 (WeightDecay)', 'Dropout', 'LabelSmoothing', 'BatchNorm+Dropout']
Optimizers:   ['SGD', 'SGD+Momentum', 'Adam', 'AdamW', 'RMSprop']
Phase 1 optimizer: Adam
CV design: 5 folds × 3 seeds = 15 fits/config
Untouched test fraction: 15%
Total model-selection fits: 165
USE_LOCAL_CSV: False


## 2. Data Loading (Kaggle API)

### One-time credential setup

The dataset is downloaded automatically via the Kaggle API. You need credentials once:

**Option A - OAuth (recommended)**
```powershell
.venv\Scripts\kaggle.exe auth login
```

**Option B - API token file**
1. Go to [kaggle.com/settings/api](https://www.kaggle.com/settings/api) → *Generate New Token*
2. Place `kaggle.json` at `~/.kaggle/kaggle.json`

**Option C - `KaggleToken.py` (local dev shortcut, git-ignored)**
```python
KAGGLE_API_TOKEN = "KGAT_xxxxxxxxxxxxxxxx"
```
The cell below reads it automatically if the file exists.

**Credential priority order:**  `KaggleToken.py` → `KAGGLE_API_TOKEN` env var → `kaggle.json` → `access_token`

In [3]:
DATA_DIR     = Path('./data')
DATA_DIR.mkdir(exist_ok=True)
DATASET_SLUG = 'ineubytes/heart-disease-dataset'
CSV_FILE     = DATA_DIR / 'heart.csv'

if USE_LOCAL_CSV:
    # ── Local-only mode ───────────────────────────────────────────────
    if not CSV_FILE.exists():
        raise FileNotFoundError(
            f'\nUSE_LOCAL_CSV = True but file not found: {CSV_FILE}\n'
            'Download heart.csv manually from:\n'
            f'  https://www.kaggle.com/datasets/{DATASET_SLUG}\n'
            'and place it at:  data/heart.csv'
        )
    print(f'\u2713 USE_LOCAL_CSV = True - loading from {CSV_FILE}')

else:
    # ── Kaggle credential resolution - multi-method, priority order ───
    #   1. KaggleToken.py  (git-ignored local file)
    #   2. KAGGLE_API_TOKEN environment variable
    #   3. ~/.kaggle/kaggle.json
    #   4. ~/.kaggle/access_token
    _token_file = Path('KaggleToken.py')
    _kaggle_dir = Path.home() / '.kaggle'
    _cred_found = False

    if _token_file.exists():
        import importlib.util as _ilu
        _spec = _ilu.spec_from_file_location('KaggleToken', _token_file)
        _mod  = _ilu.module_from_spec(_spec)
        _spec.loader.exec_module(_mod)
        os.environ['KAGGLE_API_TOKEN'] = _mod.KAGGLE_API_TOKEN
        print('\u2713 Credentials loaded from KaggleToken.py')
        _cred_found = True
    elif os.environ.get('KAGGLE_API_TOKEN'):
        print('\u2713 Using KAGGLE_API_TOKEN environment variable')
        _cred_found = True
    elif (_kaggle_dir / 'kaggle.json').exists():
        print(f'\u2713 Using {_kaggle_dir / "kaggle.json"}')
        _cred_found = True
    elif (_kaggle_dir / 'access_token').exists():
        print(f'\u2713 Using {_kaggle_dir / "access_token"}')
        _cred_found = True

    if not _cred_found:
        if CSV_FILE.exists():
            print('\u26a0 No Kaggle credentials found - using existing local CSV as fallback.')
        else:
            raise EnvironmentError(
                '\nNo Kaggle credentials found and no local CSV available.\n'
                'Options:\n'
                '  A. Create KaggleToken.py:  KAGGLE_API_TOKEN = "KGAT_xxx"\n'
                '  B. Set env var:            KAGGLE_API_TOKEN=KGAT_xxx\n'
                '  C. Place kaggle.json at:   ~/.kaggle/kaggle.json\n'
                '  D. Manual download + set USE_LOCAL_CSV = True in the Configuration cell:\n'
                f'     Download from  https://www.kaggle.com/datasets/{DATASET_SLUG}\n'
                '     Place CSV at   data/heart.csv'
            )

    # ── Dataset download ──────────────────────────────────────────────
    if _cred_found and not CSV_FILE.exists():
        import kaggle
        try:
            print(f"\nDownloading '{DATASET_SLUG}' from Kaggle...")
            kaggle.api.authenticate()
            kaggle.api.dataset_download_files(DATASET_SLUG, path=str(DATA_DIR), unzip=True)
            for _f in DATA_DIR.glob('*.zip'):
                _f.unlink()
            print('Download complete.')
        except Exception as _exc:
            if CSV_FILE.exists():
                print(f'\u26a0 Kaggle download failed ({_exc}). Using existing local CSV as fallback.')
            else:
                raise EnvironmentError(
                    f'\nKaggle API error: {_exc}\n'
                    'Your token may be invalid or expired.\n'
                    'Fix options:\n'
                    '  - Regenerate token at https://www.kaggle.com/settings/api\n'
                    '  - Or: set USE_LOCAL_CSV = True and place heart.csv at data/heart.csv'
                )
    elif _cred_found:
        print(f'\nDataset already present at: {CSV_FILE}')

# ── Load & validate ───────────────────────────────────────────────
df_raw = pd.read_csv(CSV_FILE)

_expected_cols = {'age','sex','cp','trestbps','chol','fbs','restecg',
                  'thalach','exang','oldpeak','slope','ca','thal','target'}
_missing = _expected_cols - set(df_raw.columns)
assert not _missing, f'Unexpected CSV schema - missing columns: {_missing}'
assert set(df_raw['target'].unique()).issubset({0, 1}), 'target is not binary (0/1)'

print(f'\nShape : {df_raw.shape}')
print(f'Target: {df_raw["target"].value_counts().to_dict()}')
df_raw.head()

✓ Credentials loaded from KaggleToken.py

Dataset already present at: data\heart.csv

Shape : (1025, 14)
Target: {1: 526, 0: 499}


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


## 2b. Data Integrity - Deduplication

> **Critical step.** This Kaggle release of the Cleveland dataset is heavily duplicated: it reports
> 1025 rows but contains only ~302 unique patients (~70% duplicates). Splitting *before* removing
> duplicates leaks identical records across train/validation/test and produces optimistically
> inflated metrics. We deduplicate here, before any split, so results reflect true generalisation.


In [4]:
_n_before = len(df_raw)
_n_dupes  = int(df_raw.duplicated().sum())
_dup_pct  = 100 * _n_dupes / _n_before

print(f'Rows before dedup : {_n_before}')
print(f'Exact duplicates  : {_n_dupes}  ({_dup_pct:.1f}% of the data)')
print(f'Unique patients   : {_n_before - _n_dupes}')

df_raw = df_raw.drop_duplicates().reset_index(drop=True)

print(f'\nRows after dedup  : {len(df_raw)}')
print(f'Target balance    : {df_raw["target"].value_counts().to_dict()}')
if _n_dupes > 0:
    print('\n=> Deduplication done BEFORE splitting - this is essential: identical\n'
          '   rows shared between train and test would leak the test set and\n'
          '   inflate every metric. All downstream results now reflect real\n'
          '   generalisation on unique patients.')


Rows before dedup : 1025
Exact duplicates  : 723  (70.5% of the data)
Unique patients   : 302

Rows after dedup  : 302
Target balance    : {1: 164, 0: 138}

=> Deduplication done BEFORE splitting - this is essential: identical
   rows shared between train and test would leak the test set and
   inflate every metric. All downstream results now reflect real
   generalisation on unique patients.


## 3. Exploratory Data Analysis

In [5]:
# ── 1. Target distribution ────────────────────────────────────
target_counts = df_raw['target'].value_counts().reset_index()
target_counts.columns = ['target', 'count']
target_counts['label'] = target_counts['target'].map({0: 'No Disease', 1: 'Has Disease'})
target_counts['pct'] = (target_counts['count'] / target_counts['count'].sum() * 100).round(1)
target_counts['text'] = target_counts.apply(lambda r: f"{r['count']} ({r['pct']}%)", axis=1)

px.bar(
    target_counts, x='label', y='count', color='label',
    title='Target Distribution',
    color_discrete_sequence=px.colors.qualitative.Set2,
    text='text', template='plotly_white',
).update_traces(textposition='outside').update_layout(showlegend=False).show()

# ── 2. Feature–target correlation bar ─────────────────────────
# Most actionable EDA view for classification: which features drive the target?
feat_corr = (
    df_raw.corrwith(df_raw['target'])
    .drop('target')
    .sort_values(key=abs, ascending=True)
)
px.bar(
    x=feat_corr.values, y=feat_corr.index,
    orientation='h',
    title='Feature Correlation with Target (Pearson r)',
    color=feat_corr.values,
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0,
    template='plotly_white',
    labels={'x': 'Pearson r with target', 'y': 'Feature'},
).update_layout(coloraxis_showscale=False, height=450).show()

# ── 3. Correlation heatmap (lower triangle only) ───────────────
corr = df_raw.corr(numeric_only=True)
corr_display = corr.mask(np.triu(np.ones(corr.shape, dtype=bool), k=1))
px.imshow(
    corr_display, text_auto='.2f',
    title='Feature Correlation Matrix (lower triangle)',
    color_continuous_scale='RdBu_r', zmin=-1, zmax=1, template='plotly_white',
    aspect='auto',
).show()

# ── 4. Box plots: class separation per feature ────────────────
# Box plots reveal median shift and IQR overlap between classes -
# the primary visual signal for each feature's predictive power.
numeric_cols = [c for c in df_raw.columns if c != 'target']
fig_box = make_subplots(rows=3, cols=5, subplot_titles=numeric_cols[:13])
_class_style = [(0, 'No Disease', '#636EFA'), (1, 'Has Disease', '#EF553B')]
for i, col in enumerate(numeric_cols[:13]):
    r_idx, c_idx = divmod(i, 5)
    for t_val, label, color in _class_style:
        fig_box.add_trace(
            go.Box(
                y=df_raw[df_raw['target'] == t_val][col],
                name=label, marker_color=color,
                showlegend=(i == 0), legendgroup=label,
                boxmean=True,
            ),
            row=r_idx + 1, col=c_idx + 1,
        )
fig_box.update_layout(
    title='Feature Distributions by Class (box = median/IQR, dot = mean)',
    height=650, template='plotly_white', boxmode='group',
)
fig_box.show()

## 4. Preprocessing

In [ ]:
FEATURE_COLS = [c for c in df_raw.columns if c != 'target']
TARGET_COL   = 'target'

assert df_raw[FEATURE_COLS].isnull().sum().sum() == 0, 'Missing values found in features!'

X = df_raw[FEATURE_COLS].values.astype(np.float32)
y = df_raw[TARGET_COL].values.astype(np.float32)

# Derive INPUT_DIM from data so it stays correct if the dataset changes
INPUT_DIM = X.shape[1]

# Reserve the test set once. It is never used in CV, model selection, early
# stopping, or threshold selection.
X_dev_raw, X_test_raw, y_dev, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

# Final-model scaler is fitted on all development data only. CV fits a fresh
# scaler inside every fold to prevent preprocessing leakage.
final_scaler = StandardScaler()
X_dev  = final_scaler.fit_transform(X_dev_raw).astype(np.float32)
X_test = final_scaler.transform(X_test_raw).astype(np.float32)

def to_loader(X_arr, y_arr, shuffle=False, seed=SEED):
    dataset = TensorDataset(
        torch.tensor(X_arr, dtype=torch.float32),
        torch.tensor(y_arr, dtype=torch.float32),
    )
    generator = torch.Generator().manual_seed(seed) if shuffle else None
    return DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=shuffle, generator=generator
    )

train_loader = to_loader(X_dev,  y_dev,  shuffle=True, seed=SEED)
test_loader  = to_loader(X_test, y_test)

print(f'Input features: {INPUT_DIM}  ({FEATURE_COLS})')
print(f'Development: {len(X_dev)} | Untouched test: {len(X_test)}')
print(f'Class balance - Dev: {y_dev.mean():.3f}  Test: {y_test.mean():.3f}')
print('Scaling policy: fit within each CV fold; final scaler fit on development data only.')

## 5. Model Factory & Training Engine

### Architecture rationale
A shallow two-hidden-layer MLP is deliberately conservative for 13 features and only 302 unique
patients. A larger network would increase variance and overfitting risk without evidence of additional
signal. Regularization is injected through the architecture (Dropout, BatchNorm), objective (L1,
label smoothing), or optimizer (L2 weight decay).

### Leakage-safe resampling
Each CV fold fits its own scaler on fold-training observations. Early stopping uses fold-validation
loss, and the best fold state is restored before metrics are recorded. Five stratified folds across
three seeds produce 15 estimates per configuration.

### Learning-rate schedule
All experiments use `CosineAnnealingLR`, holding the scheduling policy constant across candidates.

In [ ]:
# ─────────────────────────────────────────────
#  Model Factory
# ─────────────────────────────────────────────

class MLP(nn.Module):
    """
    Configurable two-hidden-layer MLP for binary classification.
    Architecture adapts based on reg_config['type']:
      - none / l1 / l2  : plain linear + ReLU stack
      - dropout          : Dropout after each hidden activation
      - label_smooth     : same as none (smoothing applied in loss)
      - bn_dropout       : BatchNorm before each linear + Dropout after activation
    """
    def __init__(self, input_dim, hidden1, hidden2, reg_config):
        super().__init__()
        t      = reg_config.get('type', 'none')
        p      = reg_config.get('rate',  0.0)
        use_bn = (t == 'bn_dropout')
        use_dp = (t in ('dropout', 'bn_dropout'))
        layers = []
        if use_bn:
            layers.append(nn.BatchNorm1d(input_dim))
        layers.append(nn.Linear(input_dim, hidden1))
        layers.append(nn.ReLU())
        if use_dp:
            layers.append(nn.Dropout(p=p))
        if use_bn:
            layers.append(nn.BatchNorm1d(hidden1))
        layers.append(nn.Linear(hidden1, hidden2))
        layers.append(nn.ReLU())
        if use_dp:
            layers.append(nn.Dropout(p=p * 0.5))
        layers.append(nn.Linear(hidden2, 1))
        self.net = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x).squeeze(-1)


# ─────────────────────────────────────────────
#  Optimizer Factory
# ─────────────────────────────────────────────

def build_optimizer(model, opt_cfg, reg_cfg):
    """Weight decay uses explicit None-check so weight_decay=0.0 is respected."""
    wd_reg = reg_cfg.get('weight_decay')
    wd_opt = opt_cfg.get('weight_decay')
    wd     = wd_reg if wd_reg is not None else (wd_opt if wd_opt is not None else 0.0)
    lr     = opt_cfg['lr']
    t      = opt_cfg['type']
    if t == 'sgd':
        return torch.optim.SGD(
            model.parameters(), lr=lr,
            momentum=opt_cfg.get('momentum', 0.0),
            weight_decay=wd,
        )
    if t == 'adam':
        return torch.optim.Adam(model.parameters(),   lr=lr, weight_decay=wd)
    if t == 'adamw':
        return torch.optim.AdamW(model.parameters(),  lr=lr, weight_decay=wd)
    if t == 'rmsprop':
        return torch.optim.RMSprop(model.parameters(), lr=lr, weight_decay=wd)
    raise ValueError(f'Unknown optimizer type: {t!r}')


# ─────────────────────────────────────────────
#  Label Smoothing Loss
# ─────────────────────────────────────────────

class LabelSmoothingBCE(nn.Module):
    """
    Binary cross-entropy with label smoothing.
    Softens hard 0/1 targets to (s/2, 1-s/2), reducing overconfident logits.
    Particularly useful in small medical datasets where label noise is high.
    """
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, logits, targets):
        targets_s = targets * (1.0 - self.smoothing) + 0.5 * self.smoothing
        return F.binary_cross_entropy_with_logits(logits, targets_s)


def get_criterion(reg_cfg):
    if reg_cfg.get('type') == 'label_smooth':
        return LabelSmoothingBCE(smoothing=reg_cfg.get('smoothing', 0.1))
    return nn.BCEWithLogitsLoss()


def l1_penalty(model, lambda_):
    return lambda_ * sum(p.abs().sum() for p in model.parameters() if p.requires_grad)


print('Model factory defined.')

In [ ]:
# -----------------------------------------------------------------------------
# Training, Evaluation & Cross-Validation
# -----------------------------------------------------------------------------

def train_epoch(model, loader, optimizer, criterion, reg_cfg, device):
    model.train()
    total_loss = 0.0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_b), y_b)
        if reg_cfg.get('type') == 'l1':
            loss = loss + l1_penalty(model, reg_cfg['l1_lambda'])
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(X_b)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def get_probs_targets(model, loader, device):
    model.eval()
    all_probs, all_targets = [], []
    for X_b, y_b in loader:
        logits = model(X_b.to(device))
        all_probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_targets.extend(y_b.numpy())
    return np.asarray(all_probs), np.asarray(all_targets)


@torch.no_grad()
def eval_loader(model, loader, criterion, reg_cfg, device):
    model.eval()
    total_loss = 0.0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        total_loss += criterion(model(X_b), y_b).item() * len(X_b)
    probs, targets = get_probs_targets(model, loader, device)
    preds = (probs >= 0.5).astype(int)
    return {
        'loss': total_loss / len(loader.dataset),
        'accuracy': float(accuracy_score(targets, preds)),
        'f1': float(f1_score(targets, preds, zero_division=0)),
        'roc_auc': float(roc_auc_score(targets, probs)),
    }


def _run_fold(X_train_raw, y_train, X_val_raw, y_val, reg_cfg, opt_cfg,
              device, seed, epochs=EPOCHS, patience=EARLY_STOP_PAT):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
    X_val = scaler.transform(X_val_raw).astype(np.float32)
    fold_train_loader = to_loader(X_train, y_train, shuffle=True, seed=seed)
    fold_val_loader = to_loader(X_val, y_val)

    model = MLP(INPUT_DIM, HIDDEN1, HIDDEN2, reg_cfg).to(device)
    optimizer = build_optimizer(model, opt_cfg, reg_cfg)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = get_criterion(reg_cfg)

    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': [], 'val_roc_auc': []}
    best_val_loss, best_state, best_val_metrics = float('inf'), None, None
    no_improve = 0

    for _ in range(epochs):
        train_loss = train_epoch(model, fold_train_loader, optimizer, criterion, reg_cfg, device)
        val_metrics = eval_loader(model, fold_val_loader, criterion, reg_cfg, device)
        scheduler.step()
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['accuracy'])
        history['val_f1'].append(val_metrics['f1'])
        history['val_roc_auc'].append(val_metrics['roc_auc'])

        if val_metrics['loss'] < best_val_loss:
            best_val_loss = val_metrics['loss']
            best_state = {key: value.detach().clone() for key, value in model.state_dict().items()}
            best_val_metrics = dict(val_metrics)
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    model.load_state_dict(best_state)
    val_probs, val_targets = get_probs_targets(model, fold_val_loader, device)
    return {
        'metrics': best_val_metrics,
        'history': history,
        'stopped_epoch': len(history['train_loss']),
        'val_probs': val_probs,
        'val_targets': val_targets,
    }


def _mean_history(histories):
    mean_history = {}
    for key in histories[0]:
        max_len = max(len(history[key]) for history in histories)
        padded = np.full((len(histories), max_len), np.nan)
        for row, history in enumerate(histories):
            values = history[key]
            padded[row, :len(values)] = values
        mean_history[key] = np.nanmean(padded, axis=0).tolist()
    return mean_history


def run_cv_experiment(reg_cfg, opt_cfg, device):
    fold_metrics, histories, stopped_epochs = [], [], []
    oof_sum = np.zeros(len(y_dev), dtype=float)
    oof_count = np.zeros(len(y_dev), dtype=int)

    for cv_seed in CV_SEEDS:
        splitter = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=cv_seed)
        for fold_idx, (train_idx, val_idx) in enumerate(splitter.split(X_dev_raw, y_dev), start=1):
            fold_seed = cv_seed * 100 + fold_idx
            fold_result = _run_fold(
                X_dev_raw[train_idx], y_dev[train_idx],
                X_dev_raw[val_idx], y_dev[val_idx],
                reg_cfg, opt_cfg, device, fold_seed,
            )
            fold_metrics.append(fold_result['metrics'])
            histories.append(fold_result['history'])
            stopped_epochs.append(fold_result['stopped_epoch'])
            oof_sum[val_idx] += fold_result['val_probs']
            oof_count[val_idx] += 1

    metric_frame = pd.DataFrame(fold_metrics)
    assert np.all(oof_count == len(CV_SEEDS)), 'Each development sample must receive one OOF prediction per seed.'
    return {
        'cv_metrics': metric_frame.mean().to_dict(),
        'cv_std': metric_frame.std(ddof=1).to_dict(),
        'history': _mean_history(histories),
        'stopped_epoch': float(np.mean(stopped_epochs)),
        'fold_metrics': metric_frame,
        'oof_probs': oof_sum / oof_count,
        'oof_targets': y_dev.copy(),
    }


# -----------------------------------------------------------------------------
# Plotly table helper
# -----------------------------------------------------------------------------

def make_table_fig(df, highlight_cols=(), title=''):
    """Build a compact Plotly table with wrapped labels and accessible contrast."""
    import textwrap
    import plotly.colors as _pc

    df = df.copy().reset_index(drop=True)
    row_count = len(df)
    cell_values, cell_colors, font_colors, column_widths = [], [], [], []

    def _wrap(value, width=16):
        text = str(value)
        return '<br>'.join(textwrap.wrap(text, width=width, break_long_words=False)) or text

    def _header(column):
        words = str(column).replace('_', ' ').upper().split()
        if len(words) <= 2:
            return ' '.join(words)
        midpoint = (len(words) + 1) // 2
        return ' '.join(words[:midpoint]) + '<br>' + ' '.join(words[midpoint:])

    for column in df.columns:
        series = df[column]
        is_numeric = pd.api.types.is_numeric_dtype(series)

        if is_numeric:
            display_values = [f'{value:.4f}' if pd.notna(value) else '' for value in series]
            column_widths.append(88)
        else:
            display_values = [_wrap(value) for value in series]
            longest = max([len(str(column)), *(len(str(value)) for value in series)], default=12)
            column_widths.append(min(170, max(90, longest * 8)))
        cell_values.append(display_values)

        if column in highlight_cols and is_numeric:
            values = series.astype(float).to_numpy()
            finite = np.isfinite(values)
            vmin = float(np.min(values[finite])) if finite.any() else 0.0
            vmax = float(np.max(values[finite])) if finite.any() else 1.0
            denominator = vmax - vmin if vmax > vmin else 1.0
            normalized = np.where(finite, (values - vmin) / denominator, 0.0)
            fills = [_pc.sample_colorscale('YlGn', float(value))[0] for value in normalized]
            text_colors = ['white' if value >= 0.62 else '#17212b' for value in normalized]
        else:
            fills = ['#ffffff' if row % 2 == 0 else '#f5f7fa' for row in range(row_count)]
            text_colors = ['#17212b'] * row_count
        cell_colors.append(fills)
        font_colors.append(text_colors)

    figure = go.Figure(data=[go.Table(
        columnwidth=column_widths,
        header=dict(
            values=[f'<b>{_header(column)}</b>' for column in df.columns],
            fill_color='#2c5f8a',
            font=dict(color='white', size=11),
            align='center',
            height=38,
        ),
        cells=dict(
            values=cell_values,
            fill_color=cell_colors,
            font=dict(color=font_colors, size=11),
            align=['left' if not pd.api.types.is_numeric_dtype(df[column]) else 'center' for column in df.columns],
            height=38,
        ),
    )])
    figure.update_layout(
        title=title,
        template='plotly_white',
        margin=dict(l=8, r=8, t=48 if title else 12, b=8),
        height=max(170, 92 + row_count * 38),
    )
    return figure


print('Multi-seed stratified CV engine ready.')

## 6. Phase 1 - Multi-Seed CV Regularizer Ablation

Each regularizer is evaluated with the same fixed optimizer across 5 stratified folds and three
independent split/initialization seeds (15 fits per configuration). Selection uses mean out-of-fold
ROC-AUC; standard deviations quantify sensitivity to sampling and initialization. The test set remains untouched.

In [ ]:
phase1_opt_cfg = next(o for o in OPTIMIZERS if o['name'] == PHASE1_OPTIMIZER_NAME)
phase1_raw = []

print(f'Phase 1: {CV_FOLDS}-fold CV × {len(CV_SEEDS)} seeds | optimizer = {PHASE1_OPTIMIZER_NAME}')
print('=' * 78)

for reg in REGULARIZERS:
    print(f'  {reg["name"]:<22}', end=' ', flush=True)
    result = run_cv_experiment(reg, phase1_opt_cfg, DEVICE)
    record = {
        'phase': 'Phase 1',
        'regularizer': reg['name'],
        'optimizer': PHASE1_OPTIMIZER_NAME,
        **{f'cv_{key}': value for key, value in result['cv_metrics'].items()},
        **{f'cv_std_{key}': value for key, value in result['cv_std'].items()},
        'stopped_epoch': result['stopped_epoch'],
        '_history': result['history'],
        '_fold_metrics': result['fold_metrics'],
        '_oof_probs': result['oof_probs'],
        '_oof_targets': result['oof_targets'],
    }
    phase1_raw.append(record)
    print(f'AUC={record["cv_roc_auc"]:.4f} ± {record["cv_std_roc_auc"]:.4f}  '
          f'F1={record["cv_f1"]:.4f} ± {record["cv_std_f1"]:.4f}  '
          f'ep={record["stopped_epoch"]:.1f}')

print('\nPhase 1 complete. No test-set observations were evaluated.')

### Phase 1 Results - Visualizations

In [ ]:
df_p1 = pd.DataFrame(
    [{key: value for key, value in record.items() if not key.startswith('_')} for record in phase1_raw]
)

df_melt = df_p1.melt(
    id_vars='regularizer', value_vars=list(METRIC_MAP.keys()),
    var_name='metric_key', value_name='score',
)
df_melt['metric'] = df_melt['metric_key'].map(METRIC_MAP)
df_melt['std'] = [
    df_p1.loc[df_p1['regularizer'] == row.regularizer, f'cv_std_{row.metric_key[3:]}'].iloc[0]
    for row in df_melt.itertuples()
]

fig_p1_bar = px.bar(
    df_melt, x='regularizer', y='score', color='metric', error_y='std',
    barmode='group',
    title=f'Phase 1: Multi-Seed CV Regularizer Comparison (optimizer = {PHASE1_OPTIMIZER_NAME})',
    labels={'score': 'Mean CV score', 'regularizer': 'Regularizer'},
    color_discrete_sequence=px.colors.qualitative.Plotly,
    template='plotly_white', text_auto='.3f',
)
fig_p1_bar.update_layout(yaxis=dict(range=[0.4, 1.05]))
fig_p1_bar.update_traces(textposition='outside')
fig_p1_bar.show()

# ── Stopped-epoch bar ────────────────────────────────────────
px.bar(
    df_p1, x='regularizer', y='stopped_epoch',
    title='Phase 1: Mean Epochs Until Early Stopping (15 fits/config)',
    color='regularizer', template='plotly_white', text_auto='.1f',
).update_traces(textposition='outside').show()

In [ ]:
# Mean learning trajectories across all folds and seeds. Curves are descriptive;
# scalar CV metrics above remain the basis for model selection.
fig_lc = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Mean Train Loss', 'Mean Validation Loss', 'Mean Overfitting Gap'),
)
colors_p1 = px.colors.qualitative.Plotly
for idx, record in enumerate(phase1_raw):
    history = record['_history']
    epochs = list(range(1, len(history['train_loss']) + 1))
    color = colors_p1[idx % len(colors_p1)]
    name = record['regularizer']
    gap = [val - train for val, train in zip(history['val_loss'], history['train_loss'])]
    fig_lc.add_trace(go.Scatter(x=epochs, y=history['train_loss'], name=name,
                                line=dict(color=color), legendgroup=name), row=1, col=1)
    fig_lc.add_trace(go.Scatter(x=epochs, y=history['val_loss'], name=name,
                                line=dict(color=color, dash='dot'),
                                legendgroup=name, showlegend=False), row=1, col=2)
    fig_lc.add_trace(go.Scatter(x=epochs, y=gap, name=name,
                                line=dict(color=color, dash='dashdot'),
                                legendgroup=name, showlegend=False), row=1, col=3)

fig_lc.add_hline(y=0, line_dash='dash', line_color='grey', row=1, col=3)
fig_lc.update_layout(title='Phase 1: Mean Learning Curves Across CV Fits',
                     template='plotly_white', height=420)
fig_lc.show()

# ── Val ROC-AUC over epochs ───────────────────────────────────
fig_auc = go.Figure()
for idx, record in enumerate(phase1_raw):
    history = record['_history']
    epochs = list(range(1, len(history['val_roc_auc']) + 1))
    fig_auc.add_trace(go.Scatter(
        x=epochs, y=history['val_roc_auc'], name=record['regularizer'],
        line=dict(color=colors_p1[idx % len(colors_p1)]),
    ))
fig_auc.update_layout(title='Phase 1: Mean Validation ROC-AUC Across CV Fits',
                      xaxis_title='Epoch', yaxis_title='ROC-AUC', template='plotly_white')
fig_auc.show()

## 7. Automatic Best-Regularizer Selection

The best regularizer is selected by **mean multi-seed CV ROC-AUC**. The standard deviation is reported
as a stability diagnostic. The untouched test set is not evaluated during selection.

In [ ]:
PRACTICAL_AUC_MARGIN = 0.01

best_p1_idx = df_p1['cv_roc_auc'].idxmax()
best_p1_row = df_p1.loc[best_p1_idx]
BEST_REG_NAME = best_p1_row['regularizer']
best_reg_cfg = next(reg for reg in REGULARIZERS if reg['name'] == BEST_REG_NAME)

p1_auc_gap = float(best_p1_row['cv_roc_auc'] - df_p1['cv_roc_auc'].nlargest(2).iloc[-1])
p1_near_best = df_p1.loc[
    df_p1['cv_roc_auc'] >= best_p1_row['cv_roc_auc'] - PRACTICAL_AUC_MARGIN,
    'regularizer',
].tolist()

print(f'Best regularizer : {BEST_REG_NAME}')
print(f'CV ROC-AUC       : {best_p1_row["cv_roc_auc"]:.4f} ± {best_p1_row["cv_std_roc_auc"]:.4f}')
print(f'CV F1-Score      : {best_p1_row["cv_f1"]:.4f} ± {best_p1_row["cv_std_f1"]:.4f}')
print(f'CV Accuracy      : {best_p1_row["cv_accuracy"]:.4f} ± {best_p1_row["cv_std_accuracy"]:.4f}')
print(f'AUC lead over #2 : {p1_auc_gap:.4f}')
print(f'Within {PRACTICAL_AUC_MARGIN:.2f} AUC of best: {p1_near_best}')
print(f'\n=> Phase 2 will use {BEST_REG_NAME!r} as fixed regularizer.')
print('The near-best set is descriptive; the strict mean-AUC rule still determines selection.')

p1_display = (
    df_p1.sort_values('cv_roc_auc', ascending=False)
    [['regularizer', 'cv_roc_auc', 'cv_std_roc_auc', 'cv_f1', 'cv_std_f1',
      'cv_accuracy', 'cv_std_accuracy', 'stopped_epoch']]
    .rename(columns={
        'regularizer': 'Regularizer',
        'cv_roc_auc': 'ROC-AUC',
        'cv_std_roc_auc': 'AUC SD',
        'cv_f1': 'F1',
        'cv_std_f1': 'F1 SD',
        'cv_accuracy': 'Accuracy',
        'cv_std_accuracy': 'Accuracy SD',
        'stopped_epoch': 'Mean stop epoch',
    })
)
make_table_fig(
    p1_display,
    highlight_cols=('ROC-AUC', 'F1', 'Accuracy'),
    title='Phase 1: Multi-Seed CV Results (sorted by mean ROC-AUC)',
).show()

## 8. Phase 2 - Multi-Seed CV Optimizer Ablation

All optimizers are evaluated with the Phase 1 winner, using the same folds, seeds, early-stopping rule,
and fold-local scaling. This holds the resampling design constant across optimizer comparisons.

In [ ]:
phase2_raw = []

print(f'Phase 2: {CV_FOLDS}-fold CV × {len(CV_SEEDS)} seeds | regularizer = {BEST_REG_NAME}')
print('=' * 78)

for opt in OPTIMIZERS:
    print(f'  {opt["name"]:<18}', end=' ', flush=True)
    result = run_cv_experiment(best_reg_cfg, opt, DEVICE)
    record = {
        'phase': 'Phase 2',
        'regularizer': BEST_REG_NAME,
        'optimizer': opt['name'],
        **{f'cv_{key}': value for key, value in result['cv_metrics'].items()},
        **{f'cv_std_{key}': value for key, value in result['cv_std'].items()},
        'stopped_epoch': result['stopped_epoch'],
        '_history': result['history'],
        '_fold_metrics': result['fold_metrics'],
        '_oof_probs': result['oof_probs'],
        '_oof_targets': result['oof_targets'],
    }
    phase2_raw.append(record)
    print(f'AUC={record["cv_roc_auc"]:.4f} ± {record["cv_std_roc_auc"]:.4f}  '
          f'F1={record["cv_f1"]:.4f} ± {record["cv_std_f1"]:.4f}  '
          f'ep={record["stopped_epoch"]:.1f}')

print('\nPhase 2 complete. No test-set observations were evaluated.')

### Phase 2 Results - Visualizations

In [ ]:
df_p2 = pd.DataFrame(
    [{key: value for key, value in record.items() if not key.startswith('_')} for record in phase2_raw]
)

df_melt2 = df_p2.melt(
    id_vars='optimizer', value_vars=list(METRIC_MAP.keys()),
    var_name='metric_key', value_name='score',
)
df_melt2['metric'] = df_melt2['metric_key'].map(METRIC_MAP)
df_melt2['std'] = [
    df_p2.loc[df_p2['optimizer'] == row.optimizer, f'cv_std_{row.metric_key[3:]}'].iloc[0]
    for row in df_melt2.itertuples()
]

fig_p2_bar = px.bar(
    df_melt2, x='optimizer', y='score', color='metric', error_y='std',
    barmode='group',
    title=f'Phase 2: Multi-Seed CV Optimizer Comparison (regularizer = {BEST_REG_NAME})',
    labels={'score': 'Mean CV score', 'optimizer': 'Optimizer'},
    color_discrete_sequence=px.colors.qualitative.Plotly,
    template='plotly_white', text_auto='.3f',
)
fig_p2_bar.update_layout(yaxis=dict(range=[0.4, 1.05]))
fig_p2_bar.update_traces(textposition='outside')
fig_p2_bar.show()

# ── Learning curves + overfitting gap ────────────────────────
fig_lc2 = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Mean Train Loss', 'Mean Validation Loss', 'Mean Overfitting Gap'),
)
colors_p2 = px.colors.qualitative.Safe
for idx, record in enumerate(phase2_raw):
    history = record['_history']
    epochs = list(range(1, len(history['train_loss']) + 1))
    color = colors_p2[idx % len(colors_p2)]
    name = record['optimizer']
    gap = [val - train for val, train in zip(history['val_loss'], history['train_loss'])]
    fig_lc2.add_trace(go.Scatter(x=epochs, y=history['train_loss'], name=name,
                                 line=dict(color=color), legendgroup=name), row=1, col=1)
    fig_lc2.add_trace(go.Scatter(x=epochs, y=history['val_loss'], name=name,
                                 line=dict(color=color, dash='dot'),
                                 legendgroup=name, showlegend=False), row=1, col=2)
    fig_lc2.add_trace(go.Scatter(x=epochs, y=gap, name=name,
                                 line=dict(color=color, dash='dashdot'),
                                 legendgroup=name, showlegend=False), row=1, col=3)

fig_lc2.add_hline(y=0, line_dash='dash', line_color='grey', row=1, col=3)
fig_lc2.update_layout(title='Phase 2: Mean Learning Curves Across CV Fits',
                      template='plotly_white', height=420)
fig_lc2.show()

# ── Val ROC-AUC over epochs ───────────────────────────────────
fig_auc2 = go.Figure()
for idx, record in enumerate(phase2_raw):
    history = record['_history']
    epochs = list(range(1, len(history['val_roc_auc']) + 1))
    fig_auc2.add_trace(go.Scatter(
        x=epochs, y=history['val_roc_auc'], name=record['optimizer'],
        line=dict(color=colors_p2[idx % len(colors_p2)]),
    ))
fig_auc2.update_layout(title='Phase 2: Mean Validation ROC-AUC Across CV Fits',
                       xaxis_title='Epoch', yaxis_title='ROC-AUC', template='plotly_white')
fig_auc2.show()

## 9. Combined Results & Cross-Phase Analysis

In [ ]:
df_all = pd.DataFrame(
    [{key: value for key, value in record.items() if not key.startswith('_')}
     for record in phase1_raw + phase2_raw]
).sort_values('cv_roc_auc', ascending=False).reset_index(drop=True)

all_display = (
    df_all[['phase', 'regularizer', 'optimizer', 'cv_roc_auc', 'cv_std_roc_auc',
            'cv_f1', 'cv_accuracy', 'stopped_epoch']]
    .rename(columns={
        'phase': 'Phase',
        'regularizer': 'Regularizer',
        'optimizer': 'Optimizer',
        'cv_roc_auc': 'ROC-AUC',
        'cv_std_roc_auc': 'AUC SD',
        'cv_f1': 'F1',
        'cv_accuracy': 'Accuracy',
        'stopped_epoch': 'Mean stop epoch',
    })
)
make_table_fig(
    all_display,
    highlight_cols=('ROC-AUC', 'F1', 'Accuracy'),
    title='All Experiments (sorted by mean multi-seed CV ROC-AUC)',
).show()

In [ ]:
# ── Parallel coordinates ────────────────────────────────────
df_pc = df_all.copy()
reg_cats = {value: idx for idx, value in enumerate(df_pc['regularizer'].unique())}
opt_cats = {value: idx for idx, value in enumerate(df_pc['optimizer'].unique())}
df_pc['reg_idx'] = df_pc['regularizer'].map(reg_cats)
df_pc['opt_idx'] = df_pc['optimizer'].map(opt_cats)

go.Figure(go.Parcoords(
    line=dict(color=df_pc['cv_roc_auc'], colorscale='Viridis', showscale=True,
              colorbar=dict(title='Mean CV AUC')),
    dimensions=[
        dict(label='Regularizer', values=df_pc['reg_idx'],
             tickvals=list(reg_cats.values()), ticktext=list(reg_cats.keys())),
        dict(label='Optimizer', values=df_pc['opt_idx'],
             tickvals=list(opt_cats.values()), ticktext=list(opt_cats.keys())),
        dict(label='Mean CV ROC-AUC', values=df_pc['cv_roc_auc']),
        dict(label='AUC SD', values=df_pc['cv_std_roc_auc']),
        dict(label='Mean CV F1', values=df_pc['cv_f1']),
        dict(label='Mean CV Accuracy', values=df_pc['cv_accuracy']),
        dict(label='Mean Stop Epoch', values=df_pc['stopped_epoch']),
    ],
)).update_layout(title='Parallel Coordinates: Multi-Seed CV Experiments',
                 template='plotly_white', height=500).show()

# ── ROC-AUC ranking bar ──────────────────────────────────
df_rank = df_all.copy()
df_rank['run'] = df_rank['regularizer'] + ' / ' + df_rank['optimizer']
px.bar(
    df_rank.sort_values('cv_roc_auc'),
    x='cv_roc_auc', y='run', color='phase', error_x='cv_std_roc_auc',
    orientation='h', title='All Runs Ranked by Mean CV ROC-AUC',
    labels={'cv_roc_auc': 'Mean CV ROC-AUC', 'run': 'Regularizer / Optimizer'},
    template='plotly_white', text_auto='.4f',
).update_layout(xaxis=dict(range=[0.5, 1.0])).show()

## 10. Best Configuration Selection

In [ ]:
# The optimizer is selected explicitly from Phase 2. The combined table is for
# presentation only and must not re-enter the sequential selection procedure.
best_idx = df_p2['cv_roc_auc'].idxmax()
best_row = df_p2.loc[best_idx]
FINAL_REG_NAME = BEST_REG_NAME
FINAL_OPT_NAME = best_row['optimizer']
final_reg_cfg = next(reg for reg in REGULARIZERS if reg['name'] == FINAL_REG_NAME)
final_opt_cfg = next(opt for opt in OPTIMIZERS if opt['name'] == FINAL_OPT_NAME)

final_record = next(
    record for record in phase2_raw
    if record['optimizer'] == FINAL_OPT_NAME
)
final_oof_probs = final_record['_oof_probs']
final_oof_targets = final_record['_oof_targets']

p2_auc_gap = float(best_row['cv_roc_auc'] - df_p2['cv_roc_auc'].nlargest(2).iloc[-1])
p2_near_best = df_p2.loc[
    df_p2['cv_roc_auc'] >= best_row['cv_roc_auc'] - PRACTICAL_AUC_MARGIN,
    'optimizer',
].tolist()

print('Best configuration (sequential multi-seed CV selection):')
print(f'  Regularizer : {FINAL_REG_NAME}  [selected in Phase 1]')
print(f'  Optimizer   : {FINAL_OPT_NAME}  [selected from Phase 2 only]')
print(f'  CV ROC-AUC  : {best_row["cv_roc_auc"]:.4f} ± {best_row["cv_std_roc_auc"]:.4f}')
print(f'  CV F1-Score : {best_row["cv_f1"]:.4f} ± {best_row["cv_std_f1"]:.4f}')
print(f'  CV Accuracy : {best_row["cv_accuracy"]:.4f} ± {best_row["cv_std_accuracy"]:.4f}')
print(f'  AUC lead    : {p2_auc_gap:.4f} over the second-ranked optimizer')
print(f'  Near-best   : {p2_near_best} (within {PRACTICAL_AUC_MARGIN:.2f} AUC)')
print(f'  OOF coverage: {len(final_oof_probs)}/{len(y_dev)} development patients')
print('Small mean differences relative to CV variability do not establish optimizer superiority.')

## 11. Final Model - Stochastic Weight Averaging (SWA)

After CV-based configuration selection, the final model is trained once on **all development data**.
SWA averages late-training parameter snapshots to favor a flatter solution. The untouched test set is
used only after training. Because CV has already supplied uncertainty estimates, SWA is evaluated as
a final-training strategy rather than another model-selection stage.

In [ ]:
SWA_EPOCHS   = EPOCHS
SWA_LR       = 5e-4
SWA_START_EP = int(SWA_EPOCHS * SWA_START_FRAC)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
model_base = MLP(INPUT_DIM, HIDDEN1, HIDDEN2, final_reg_cfg).to(DEVICE)
opt_base   = build_optimizer(model_base, final_opt_cfg, final_reg_cfg)
swa_model  = AveragedModel(model_base)
swa_sched  = SWALR(opt_base, swa_lr=SWA_LR, anneal_epochs=10)
criterion  = get_criterion(final_reg_cfg)
cos_sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt_base, T_max=SWA_START_EP)

swa_history = {'train_loss': [], 'development_loss': []}

print(f'SWA retraining on all {len(y_dev)} development patients')
print(f'Base optimizer: {FINAL_OPT_NAME} | regularizer: {FINAL_REG_NAME}')
print(f'Cosine epochs 1-{SWA_START_EP}; SWA averaging {SWA_START_EP + 1}-{SWA_EPOCHS}')

for epoch in range(1, SWA_EPOCHS + 1):
    train_loss = train_epoch(model_base, train_loader, opt_base, criterion, final_reg_cfg, DEVICE)
    development_metrics = eval_loader(model_base, train_loader, criterion, final_reg_cfg, DEVICE)
    swa_history['train_loss'].append(train_loss)
    swa_history['development_loss'].append(development_metrics['loss'])
    if epoch <= SWA_START_EP:
        cos_sched.step()
    else:
        swa_model.update_parameters(model_base)
        swa_sched.step()

update_bn(train_loader, swa_model, device=DEVICE)
print('\nSWA training complete. BatchNorm statistics updated on development data.')

### Final Model Evaluation

In [ ]:
swa_test  = eval_loader(swa_model, test_loader, criterion, final_reg_cfg, DEVICE)
base_test = eval_loader(model_base, test_loader, criterion, final_reg_cfg, DEVICE)

comparison = pd.DataFrame([
    {'model': f'Base ({FINAL_REG_NAME} / {FINAL_OPT_NAME})', **base_test},
    {'model': f'SWA ({FINAL_REG_NAME} / {FINAL_OPT_NAME})', **swa_test},
])

print('Final untouched-test comparison:')
make_table_fig(
    comparison,
    highlight_cols=('roc_auc', 'f1', 'accuracy'),
    title='Final Model Comparison - Untouched Test Set',
).show()
print(f'CV selection estimate: ROC-AUC {best_row["cv_roc_auc"]:.4f} ± {best_row["cv_std_roc_auc"]:.4f}')
print('CV and test values are shown separately because they estimate performance on different samples.')

epochs = list(range(1, SWA_EPOCHS + 1))
fig_swa = go.Figure()
fig_swa.add_trace(go.Scatter(x=epochs, y=swa_history['train_loss'],
                             name='Optimization Loss', line=dict(color='steelblue')))
fig_swa.add_trace(go.Scatter(x=epochs, y=swa_history['development_loss'],
                             name='Development Loss (descriptive)',
                             line=dict(color='coral', dash='dot')))
fig_swa.add_vline(x=SWA_START_EP, line_dash='dash', line_color='green',
                  annotation_text='SWA averaging starts', annotation_position='top right')
fig_swa.update_layout(
    title=f'Final SWA Training - {FINAL_REG_NAME} / {FINAL_OPT_NAME}',
    xaxis_title='Epoch', yaxis_title='Loss', template='plotly_white',
)
fig_swa.show()

In [ ]:
from sklearn.metrics import roc_curve, confusion_matrix, precision_recall_curve, average_precision_score

swa_probs, swa_targets = get_probs_targets(swa_model, test_loader, DEVICE)
swa_preds = (swa_probs >= 0.5).astype(int)

# ── ROC curve ────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(swa_targets, swa_probs)
auc_val     = roc_auc_score(swa_targets, swa_probs)

fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(
    x=fpr, y=tpr,
    fill='tozeroy', fillcolor='rgba(99,110,250,0.15)',
    line=dict(color='royalblue', width=2),
    name=f'SWA Model (AUC = {auc_val:.4f})',
))
fig_roc.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], line=dict(color='grey', dash='dash'),
    name='Random Classifier',
))
fig_roc.update_layout(
    title='Final SWA Model - ROC Curve (Test Set)',
    xaxis_title='False Positive Rate', yaxis_title='True Positive Rate',
    template='plotly_white', legend=dict(x=0.6, y=0.1),
)
fig_roc.show()

# ── Precision-Recall curve ────────────────────────────────────
# More informative than ROC for medical datasets:
# recall = sensitivity (catching disease), precision = positive predictive value.
precision_vals, recall_vals, _ = precision_recall_curve(swa_targets, swa_probs)
ap = average_precision_score(swa_targets, swa_probs)
baseline_precision = swa_targets.mean()  # no-skill baseline = prevalence

fig_pr = go.Figure()
fig_pr.add_trace(go.Scatter(
    x=recall_vals, y=precision_vals,
    fill='tozeroy', fillcolor='rgba(0,204,150,0.12)',
    line=dict(color='seagreen', width=2),
    name=f'SWA Model (AP = {ap:.4f})',
))
fig_pr.add_hline(y=baseline_precision, line_dash='dash', line_color='grey',
                  annotation_text=f'No-skill baseline ({baseline_precision:.2f})',
                  annotation_position='top right')
fig_pr.update_layout(
    title='Final SWA Model - Precision-Recall Curve (Test Set)',
    xaxis_title='Recall (Sensitivity)', yaxis_title='Precision (PPV)',
    template='plotly_white', legend=dict(x=0.05, y=0.05),
    yaxis=dict(range=[0, 1.05]),
)
fig_pr.show()

# ── Confusion matrix ─────────────────────────────────────────
# For medical classification: FN (missed disease) >> FP (false alarm).
cm = confusion_matrix(swa_targets, swa_preds)
labels = ['No Disease', 'Has Disease']

fig_cm = px.imshow(
    cm,
    text_auto=True,
    x=labels, y=labels,
    color_continuous_scale='Blues',
    title='Confusion Matrix - SWA Model (Test Set)',
    labels=dict(x='Predicted', y='Actual', color='Count'),
    template='plotly_white',
    aspect='equal',
)
fig_cm.update_layout(
    xaxis_title='Predicted Label',
    yaxis_title='True Label',
    coloraxis_showscale=False,
)

tn, fp, fn, tp = cm.ravel()
print(f'Confusion Matrix:')
print(f'  True Positives  (TP): {tp}  - correctly identified disease')
print(f'  True Negatives  (TN): {tn}  - correctly identified no disease')
print(f'  False Positives (FP): {fp}  - false alarm (no disease, predicted disease)')
print(f'  False Negatives (FN): {fn}  - missed disease (disease, predicted no disease)')
print(f'  Sensitivity (Recall): {tp/(tp+fn):.4f}')
print(f'  Specificity:          {tn/(tn+fp):.4f}')
fig_cm.show()

## 12. Model Interpretability & Clinical Operating Point

Two questions a raw ROC-AUC cannot answer:

1. **Which features drive the predictions?** Permutation importance ranks features by how much the
   model's ROC-AUC degrades when each is shuffled - critical for clinical trust and sanity-checking
   against known cardiac risk factors.
2. **Is 0.5 the right decision threshold, and are the probabilities trustworthy?** In a screening
   context a missed case (false negative) is costlier than a false alarm, so we compare operating
   points (Youden's J, max-F1) and inspect probability **calibration**.


In [ ]:
# ─────────────────────────────────────────────
#  Permutation Feature Importance (SWA model, test set)
# ─────────────────────────────────────────────
# A neural net has no built-in feature importance. Permutation importance is
# model-agnostic: shuffle one feature's values (breaking its link to the target)
# and measure the resulting drop in ROC-AUC. A large drop => the model relies
# heavily on that feature. In a medical model this is essential - clinicians
# need to know *which signals* drive a prediction, not just the score.

N_REPEATS = 30

@torch.no_grad()
def _auc_from_array(model, X_arr, y_arr, device):
    model.eval()
    xb = torch.tensor(X_arr, dtype=torch.float32, device=device)
    probs = torch.sigmoid(model(xb)).cpu().numpy()
    return roc_auc_score(y_arr, probs)

_rng = np.random.default_rng(SEED)
baseline_auc = _auc_from_array(swa_model, X_test, y_test, DEVICE)

imp_means, imp_stds = [], []
for j in range(INPUT_DIM):
    drops = []
    for _ in range(N_REPEATS):
        X_perm = X_test.copy()
        _rng.shuffle(X_perm[:, j])
        drops.append(baseline_auc - _auc_from_array(swa_model, X_perm, y_test, DEVICE))
    imp_means.append(float(np.mean(drops)))
    imp_stds.append(float(np.std(drops)))

imp_df = pd.DataFrame(
    {'feature': FEATURE_COLS, 'importance': imp_means, 'std': imp_stds}
).sort_values('importance', ascending=True)

px.bar(
    imp_df, x='importance', y='feature', orientation='h', error_x='std',
    title=f'Permutation Feature Importance - SWA Model (baseline AUC = {baseline_auc:.4f})',
    labels={'importance': 'Mean ROC-AUC drop when shuffled', 'feature': 'Feature'},
    color='importance', color_continuous_scale='Reds', template='plotly_white',
).update_layout(coloraxis_showscale=False, height=450).show()

print('Top predictive features (permutation importance):')
for _, r in imp_df.sort_values('importance', ascending=False).head(5).iterrows():
    print(f'  {r["feature"]:<10} {r["importance"]:+.4f}')


In [ ]:
from sklearn.calibration import calibration_curve

# Thresholds are selected from averaged out-of-fold development predictions,
# then applied once to the untouched SWA test predictions.
_oof_fpr, _oof_tpr, _oof_thresholds = roc_curve(final_oof_targets, final_oof_probs)
thr_youden = float(_oof_thresholds[int(np.argmax(_oof_tpr - _oof_fpr))])

_grid = np.linspace(0.05, 0.95, 91)
thr_f1 = float(_grid[int(np.argmax([
    f1_score(final_oof_targets, (final_oof_probs >= threshold).astype(int), zero_division=0)
    for threshold in _grid
]))])

thr_probs, thr_targets = get_probs_targets(swa_model, test_loader, DEVICE)

def _test_metrics_at(threshold):
    preds = (thr_probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(thr_targets, preds).ravel()
    return {
        'threshold': round(float(threshold), 3),
        'sensitivity': tp / (tp + fn) if (tp + fn) else 0.0,
        'specificity': tn / (tn + fp) if (tn + fp) else 0.0,
        'precision': tp / (tp + fp) if (tp + fp) else 0.0,
        'f1': f1_score(thr_targets, preds, zero_division=0),
        'accuracy': accuracy_score(thr_targets, preds),
    }

thr_table = pd.DataFrame([
    {'strategy': 'Default (0.5)', **_test_metrics_at(0.5)},
    {'strategy': "OOF Youden's J", **_test_metrics_at(thr_youden)},
    {'strategy': 'OOF Max F1', **_test_metrics_at(thr_f1)},
])
make_table_fig(
    thr_table,
    highlight_cols=('sensitivity', 'specificity', 'f1', 'accuracy'),
    title='OOF-Selected Thresholds Applied to Untouched Test Set',
).show()

frac_pos, mean_pred = calibration_curve(thr_targets, thr_probs, n_bins=8, strategy='quantile')
fig_cal = go.Figure()
fig_cal.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines',
                             line=dict(color='grey', dash='dash'),
                             name='Perfectly calibrated'))
fig_cal.add_trace(go.Scatter(x=mean_pred, y=frac_pos, mode='lines+markers',
                             line=dict(color='indianred', width=2),
                             marker=dict(size=8), name='SWA model'))
fig_cal.update_layout(
    title='Calibration Curve - SWA Model (Untouched Test Set)',
    xaxis_title='Mean predicted probability',
    yaxis_title='Observed fraction positive',
    template='plotly_white', xaxis=dict(range=[0, 1]), yaxis=dict(range=[0, 1]),
    legend=dict(x=0.05, y=0.95),
)
fig_cal.show()

print(f"OOF Youden's J threshold : {thr_youden:.3f}")
print(f'OOF max-F1 threshold      : {thr_f1:.3f}')
print('Thresholds were selected without test labels. Their transfer to SWA is exploratory because')
print('the OOF models use the selected architecture/optimizer but do not apply SWA.')

## 13. Conclusions

All numbers referenced below are printed by the summary cell that follows, so they always describe
the run that actually produced this notebook rather than a remembered earlier run.

### Data integrity
This Kaggle release reports over a thousand rows but contains only ~302 unique patients. Removing the
exact duplicates *before* any split is the single most consequential decision in the pipeline: had they
been split across train and test, the model would have been scored on records it had already seen.
The resulting metrics are lower than a leakage-prone pipeline would report, and that is the point.

### Model selection
Neither ablation produced a decisive winner. In both phases the spread between the best and worst
candidate is small compared to the fold-and-seed standard deviation, which means the ranking is a
*reproducible selection rule*, not evidence that one method generalizes better than another on this
dataset. The near-best sets printed in Sections 7 and 10 make that explicit. The honest reading is
that with ~302 patients and 13 features, the choice of regularizer and optimizer matters less than
the amount of data available.

The sequential structure stays explicit: Phase 1 fixes the optimizer and selects a regularizer,
Phase 2 fixes that regularizer and selects an optimizer. The combined table in Section 9 is for
presentation only and never re-enters the selection procedure.

### Fair training budget
The epoch budget is set to 300. An earlier run capped at 150 showed plain SGD stopping close to the
cap while the Adam-family optimizers early-stopped far sooner, which would have confounded *optimizer
quality* with *training time allowed*. The mean stop-epoch column in the summary below documents
whether each optimizer now converges inside the budget; early stopping, not the cap, should be what
ends training.

### Final model and SWA
SWA is reported as a comparison against the base model, not as an assumed improvement. Whether it
helps here is answered by the summary cell; either outcome is a legitimate result on a test set of
this size, where a single reclassified patient shifts accuracy by roughly two percentage points.

### Interpretation limits
Permutation importance ranks how much the model *relies* on a feature, not what causes heart disease;
correlated predictors share credit. The calibration curve and the operating-point table are likewise
descriptive. Thresholds were derived from out-of-fold development predictions rather than test labels,
but their transfer remains exploratory because the out-of-fold models do not apply SWA. None of these
diagnostics were used to revise the model after the test set had been inspected.

### Remaining limitations
- Only ~302 unique observations; uncertainty is intrinsically high and every test-set statistic rests on ~46 patients.
- The two-phase design does not estimate regularizer-optimizer interactions.
- Integer-coded categorical features (`cp`, `restecg`, `slope`, `thal`) are treated as numeric; one-hot encoding deserves a separate leakage-safe experiment.
- `thal = 0` and `ca = 4` may be source-specific missing codes and need verification before recoding.
- No external validation cohort exists. This is an academic experiment, not a clinical decision system.


In [ ]:
# Every figure below is read from the executed run, so this summary cannot go stale.
p1_sorted = df_p1.sort_values('cv_roc_auc', ascending=False)
p2_sorted = df_p2.sort_values('cv_roc_auc', ascending=False)

print('DATA INTEGRITY')
print(f'  {_n_before} raw rows -> {len(df_raw)} unique patients ({_n_dupes} exact duplicates removed before splitting)')
print(f'  Development {len(y_dev)} | untouched test {len(y_test)} | positive rate {y_dev.mean():.3f} / {y_test.mean():.3f}')

print(f'\nPHASE 1 - REGULARIZER (optimizer fixed = {PHASE1_OPTIMIZER_NAME})')
for _, row in p1_sorted.iterrows():
    print(f'  {row["regularizer"]:<20} ROC-AUC {row["cv_roc_auc"]:.4f} ± {row["cv_std_roc_auc"]:.4f}')
print(f'  Selected: {BEST_REG_NAME} | lead over #2: {p1_auc_gap:.4f} | '
      f'best-to-worst spread: {p1_sorted["cv_roc_auc"].max() - p1_sorted["cv_roc_auc"].min():.4f} '
      f'vs typical fold SD {p1_sorted["cv_std_roc_auc"].mean():.4f}')

print(f'\nPHASE 2 - OPTIMIZER (regularizer fixed = {BEST_REG_NAME})')
for _, row in p2_sorted.iterrows():
    print(f'  {row["optimizer"]:<20} ROC-AUC {row["cv_roc_auc"]:.4f} ± {row["cv_std_roc_auc"]:.4f}  '
          f'mean stop epoch {row["stopped_epoch"]:5.1f} / {EPOCHS}')
print(f'  Selected: {FINAL_OPT_NAME} | lead over #2: {p2_auc_gap:.4f} | '
      f'best-to-worst spread: {p2_sorted["cv_roc_auc"].max() - p2_sorted["cv_roc_auc"].min():.4f} '
      f'vs typical fold SD {p2_sorted["cv_std_roc_auc"].mean():.4f}')

print(f'\nFINAL MODEL - {FINAL_REG_NAME} / {FINAL_OPT_NAME}, untouched test set (n = {len(y_test)})')
print(f'  Base  ROC-AUC {base_test["roc_auc"]:.4f} | F1 {base_test["f1"]:.4f} | '
      f'Accuracy {base_test["accuracy"]:.4f} | Loss {base_test["loss"]:.4f}')
print(f'  SWA   ROC-AUC {swa_test["roc_auc"]:.4f} | F1 {swa_test["f1"]:.4f} | '
      f'Accuracy {swa_test["accuracy"]:.4f} | Loss {swa_test["loss"]:.4f}')
print(f'  SWA effect: ROC-AUC {swa_test["roc_auc"] - base_test["roc_auc"]:+.4f}, '
      f'loss {swa_test["loss"] - base_test["loss"]:+.4f}')
print(f'  SWA confusion @0.50: TP {tp}  TN {tn}  FP {fp}  FN {fn}  -> '
      f'sensitivity {tp / (tp + fn):.4f}, specificity {tn / (tn + fp):.4f}, average precision {ap:.4f}')
print(f'  OOF-selected thresholds: Youden {thr_youden:.3f}, max-F1 {thr_f1:.3f}')
print(f'  Top-5 permutation importance: '
      f'{", ".join(imp_df.sort_values("importance", ascending=False).head(5)["feature"])}')
print(f'\n  CV estimate {best_row["cv_roc_auc"]:.4f} ± {best_row["cv_std_roc_auc"]:.4f} vs '
      f'test {base_test["roc_auc"]:.4f} -> optimism {best_row["cv_roc_auc"] - base_test["roc_auc"]:+.4f} ROC-AUC')
